# Gerador de Letras com LLM (GPT-2 Portuguese)

Fine-tune de GPT-2 treinado em portugues no corpus de letras brasileiras (~5.159 musicas).

## Funcionalidades

- Fine-tune do modelo GPT-2 portugues no corpus de letras
- Salva/carrega modelo do Google Drive (treina uma vez so)
- Geracao condicionada por **genero** (MPB, Trap, Sertanejo, Pagode, Arrocha)
- Controles de temperatura, top-k, top-p para variar a criatividade
- 100% gratuito (Colab free tier + T4 GPU)

## Como usar

1. Conecte uma GPU: `Runtime > Change runtime type > T4 GPU`
2. Faca upload do `training_corpus.txt` para o Google Drive (pasta `musicas_model/`)
3. Execute as celulas em ordem
4. Na primeira vez: treine o modelo (~30-60 min)
5. Nas proximas vezes: carregue direto do Drive e gere letras

---

## 1. Setup do Ambiente

In [ ]:
# Verificar GPU disponivel
!nvidia-smi

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nUsando dispositivo: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Instalar dependencias
!pip install -q transformers datasets accelerate
print("Dependencias instaladas.")

In [ ]:
import os
import re
import json
import random
import textwrap
from pathlib import Path

import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    set_seed,
)

set_seed(42)
print("Imports OK.")

## 2. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Diretorio base no Drive para salvar modelo e dados
DRIVE_BASE = '/content/drive/MyDrive/musicas_model'
os.makedirs(DRIVE_BASE, exist_ok=True)

# Caminhos
CORPUS_PATH = os.path.join(DRIVE_BASE, 'training_corpus.txt')
MODEL_SAVE_DIR = os.path.join(DRIVE_BASE, 'gpt2-letras-pt')

print(f"Diretorio base no Drive: {DRIVE_BASE}")
print(f"Corpus esperado em: {CORPUS_PATH}")
print(f"Modelo sera salvo em: {MODEL_SAVE_DIR}")

if os.path.exists(CORPUS_PATH):
    size_mb = os.path.getsize(CORPUS_PATH) / (1024 * 1024)
    print(f"\nCorpus encontrado! ({size_mb:.1f} MB)")
else:
    print(f"\nCorpus NAO encontrado em {CORPUS_PATH}")
    print("Faca upload do arquivo training_corpus.txt para a pasta musicas_model/ no Drive.")
    print("\nOu use o upload manual na celula abaixo.")

In [ ]:
# OPCIONAL: Upload manual do corpus (descomente se necessario)
# from google.colab import files
# print("Selecione o arquivo training_corpus.txt:")
# uploaded = files.upload()
# for filename in uploaded:
#     with open(CORPUS_PATH, 'wb') as f:
#         f.write(uploaded[filename])
#     print(f"Arquivo salvo em: {CORPUS_PATH}")

## 3. Preparar Corpus para Fine-Tune

O `training_corpus.txt` usa o formato:
```
### Titulo - Artista [Genero]

Letra da musica aqui...

================================================================================
```

Vamos transformar cada musica num exemplo de treino com tokens especiais para que o modelo aprenda a gerar condicionado por genero.

In [ ]:
def parse_training_corpus(filepath):
    """
    Le o training_corpus.txt e retorna lista de dicts com
    title, artist, genre, lyrics.
    """
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()

    # Separar musicas pelo divisor
    raw_songs = content.split('=' * 80)

    songs = []
    header_pattern = re.compile(
        r'###\s+(.+?)\s+-\s+(.+?)\s+\[(.+?)\]'
    )

    for raw in raw_songs:
        raw = raw.strip()
        if not raw:
            continue

        lines = raw.split('\n')
        header_line = lines[0].strip()

        match = header_pattern.match(header_line)
        if match:
            title = match.group(1).strip()
            artist = match.group(2).strip()
            genre = match.group(3).strip()
            # Letra = tudo apos o header (pular linha em branco)
            lyrics_lines = lines[1:]
            lyrics = '\n'.join(lyrics_lines).strip()
        else:
            # Fallback: tentar extrair sem o padrao exato
            title = 'Desconhecida'
            artist = 'Desconhecido'
            genre = 'MPB'
            lyrics = raw.strip()

        if len(lyrics) > 50:  # Filtrar entradas muito curtas
            songs.append({
                'title': title,
                'artist': artist,
                'genre': genre,
                'lyrics': lyrics
            })

    return songs


songs = parse_training_corpus(CORPUS_PATH)
print(f"Total de musicas carregadas: {len(songs)}")

# Distribuicao por genero
from collections import Counter
genre_counts = Counter(s['genre'] for s in songs)
print("\nDistribuicao por genero:")
for genre, count in genre_counts.most_common():
    print(f"  {genre:<12} {count:>5} musicas")

# Exemplo
print(f"\nExemplo de entrada:")
sample = songs[0]
print(f"  Titulo: {sample['title']}")
print(f"  Artista: {sample['artist']}")
print(f"  Genero: {sample['genre']}")
print(f"  Letra (primeiras 3 linhas):")
for line in sample['lyrics'].split('\n')[:3]:
    print(f"    {line}")

In [ ]:
def format_song_for_training(song):
    """
    Formata uma musica com tokens especiais para treino.

    Formato:
    <|genero|>Pagode<|titulo|>Nome da Musica<|letra|>
    Primeira linha da letra
    Segunda linha...
    <|fim|>
    """
    genre = song['genre']
    title = song['title']
    lyrics = song['lyrics'].strip()

    text = f"<|genero|>{genre}<|titulo|>{title}<|letra|>\n{lyrics}\n<|fim|>"
    return text


# Formatar todas as musicas
formatted_texts = [format_song_for_training(s) for s in songs]

print(f"Musicas formatadas: {len(formatted_texts)}")
print(f"\nExemplo formatado (primeiras 300 chars):")
print(formatted_texts[0][:300])
print("...")

## 4. Carregar Modelo Base e Tokenizer

Usamos o `pierreguillou/gpt2-small-portuguese` - um GPT-2 pre-treinado em textos em portugues (~500MB).

In [ ]:
BASE_MODEL = 'pierreguillou/gpt2-small-portuguese'

print(f"Carregando tokenizer de {BASE_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# Adicionar tokens especiais para condicionamento
SPECIAL_TOKENS = {
    'additional_special_tokens': [
        '<|genero|>',
        '<|titulo|>',
        '<|letra|>',
        '<|fim|>',
    ],
    'pad_token': '<|pad|>',
}

num_added = tokenizer.add_special_tokens(SPECIAL_TOKENS)
print(f"Tokens especiais adicionados: {num_added}")
print(f"Vocabulario total: {len(tokenizer)}")

# Verificar tokens
for tok in SPECIAL_TOKENS['additional_special_tokens']:
    tok_id = tokenizer.convert_tokens_to_ids(tok)
    print(f"  {tok} -> id {tok_id}")

In [ ]:
print(f"Carregando modelo {BASE_MODEL}...")
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL)

# Redimensionar embeddings para os novos tokens
model.resize_token_embeddings(len(tokenizer))

model = model.to(device)

# Info do modelo
total_params = sum(p.numel() for p in model.parameters())
print(f"Parametros totais: {total_params / 1e6:.1f}M")
print(f"Modelo carregado em: {device}")

## 5. Criar Dataset de Treino

In [ ]:
class LyricsDataset(Dataset):
    """
    Dataset de letras tokenizadas para fine-tune do GPT-2.
    Cada musica e truncada/padded para max_length tokens.
    """

    def __init__(self, texts, tokenizer, max_length=512):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.examples = []

        print(f"Tokenizando {len(texts)} musicas (max_length={max_length})...")

        too_long = 0
        for text in texts:
            encoding = tokenizer(
                text,
                truncation=True,
                max_length=max_length,
                padding='max_length',
                return_tensors='pt',
            )

            input_ids = encoding['input_ids'].squeeze()
            attention_mask = encoding['attention_mask'].squeeze()

            # Checar se foi truncado
            full_len = len(tokenizer.encode(text))
            if full_len > max_length:
                too_long += 1

            self.examples.append({
                'input_ids': input_ids,
                'attention_mask': attention_mask,
                'labels': input_ids.clone(),
            })

        print(f"Dataset criado: {len(self.examples)} exemplos")
        if too_long > 0:
            print(f"  ({too_long} musicas truncadas para {max_length} tokens)")

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]


# Separar treino e validacao (95/5)
random.shuffle(formatted_texts)
split_idx = int(len(formatted_texts) * 0.95)
train_texts = formatted_texts[:split_idx]
val_texts = formatted_texts[split_idx:]

print(f"Treino: {len(train_texts)} musicas")
print(f"Validacao: {len(val_texts)} musicas")

# Criar datasets
MAX_LENGTH = 512  # ~400 palavras, cobre a maioria das letras

train_dataset = LyricsDataset(train_texts, tokenizer, max_length=MAX_LENGTH)
val_dataset = LyricsDataset(val_texts, tokenizer, max_length=MAX_LENGTH)

## 6. Treinar o Modelo

Fine-tune com HuggingFace Trainer. Configurado para:
- ~3 epocas (suficiente para aprender o estilo sem overfitting)
- Learning rate baixo (5e-5) para nao destruir o conhecimento pre-treinado
- Checkpoints salvos a cada 500 steps

**Tempo estimado: 30-60 min na T4**

In [ ]:
# Diretorio local para checkpoints durante o treino
OUTPUT_DIR = '/content/gpt2-letras-checkpoints'

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,

    # Hiperparametros
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,  # Batch efetivo = 16
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=100,
    lr_scheduler_type='cosine',

    # Avaliacao e salvamento
    eval_strategy='steps',
    eval_steps=500,
    save_strategy='steps',
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',

    # Performance
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,

    # Logging
    logging_steps=50,
    report_to='none',  # Sem wandb/tensorboard

    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

print("Trainer configurado.")
print(f"  Epocas: {training_args.num_train_epochs}")
print(f"  Batch size efetivo: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  FP16: {training_args.fp16}")

In [ ]:
# ====================================================================
# TREINAR O MODELO
# Execute esta celula APENAS na primeira vez.
# Nas proximas vezes, pule para a secao "Carregar Modelo do Drive".
# ====================================================================

print("Iniciando treino...")
print("(Isso leva ~30-60 min na T4)\n")

train_result = trainer.train()

# Resultados
print("\n" + "=" * 60)
print(" TREINO FINALIZADO ".center(60, "="))
print("=" * 60)
print(f"  Loss final: {train_result.training_loss:.4f}")
print(f"  Steps totais: {train_result.global_step}")
print(f"  Tempo: {train_result.metrics.get('train_runtime', 0) / 60:.1f} min")

In [ ]:
# Avaliar no conjunto de validacao
eval_result = trainer.evaluate()
print(f"Eval loss: {eval_result['eval_loss']:.4f}")
print(f"Perplexity: {torch.exp(torch.tensor(eval_result['eval_loss'])):.2f}")

## 7. Salvar Modelo no Google Drive

Salva o modelo treinado + tokenizer no Drive para reutilizar sem retreinar.

In [ ]:
print(f"Salvando modelo em: {MODEL_SAVE_DIR}")
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

# Salvar modelo e tokenizer
trainer.save_model(MODEL_SAVE_DIR)
tokenizer.save_pretrained(MODEL_SAVE_DIR)

# Salvar metadados do treino
metadata = {
    'base_model': BASE_MODEL,
    'total_songs': len(songs),
    'genres': dict(genre_counts),
    'max_length': MAX_LENGTH,
    'epochs': training_args.num_train_epochs,
    'final_loss': train_result.training_loss,
    'eval_loss': eval_result['eval_loss'],
    'special_tokens': SPECIAL_TOKENS,
}

with open(os.path.join(MODEL_SAVE_DIR, 'training_metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

# Tamanho total
total_size = sum(
    os.path.getsize(os.path.join(MODEL_SAVE_DIR, f))
    for f in os.listdir(MODEL_SAVE_DIR)
    if os.path.isfile(os.path.join(MODEL_SAVE_DIR, f))
)
print(f"\nModelo salvo! Tamanho total: {total_size / (1024**2):.0f} MB")
print("\nNa proxima sessao, pule o treino e carregue direto do Drive.")

---

## 8. Carregar Modelo do Drive (Pular Treino)

**Use esta secao nas proximas sessoes** para carregar o modelo ja treinado sem retreinar.

Prerequisito: execute as secoes 1 (Setup) e 2 (Montar Drive) antes.

In [ ]:
# ====================================================================
# CARREGAR MODELO JA TREINADO DO DRIVE
# Use esta celula ao inves de treinar novamente.
# ====================================================================

import os
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

DRIVE_BASE = '/content/drive/MyDrive/musicas_model'
MODEL_SAVE_DIR = os.path.join(DRIVE_BASE, 'gpt2-letras-pt')

if not os.path.exists(MODEL_SAVE_DIR):
    print(f"Modelo nao encontrado em {MODEL_SAVE_DIR}")
    print("Voce precisa treinar primeiro (secoes 3-7).")
else:
    print(f"Carregando modelo de {MODEL_SAVE_DIR}...")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_SAVE_DIR)
    model = AutoModelForCausalLM.from_pretrained(MODEL_SAVE_DIR)
    model = model.to(device)
    model.eval()

    # Carregar metadados
    meta_path = os.path.join(MODEL_SAVE_DIR, 'training_metadata.json')
    if os.path.exists(meta_path):
        with open(meta_path, 'r') as f:
            metadata = json.load(f)
        print(f"\nModelo carregado com sucesso!")
        print(f"  Base: {metadata.get('base_model', '?')}")
        print(f"  Treinado em: {metadata.get('total_songs', '?')} musicas")
        print(f"  Generos: {', '.join(metadata.get('genres', {}).keys())}")
        print(f"  Loss final: {metadata.get('final_loss', '?')}")
    else:
        print("Modelo carregado com sucesso!")

    print(f"\nDispositivo: {device}")
    print("Pronto para gerar letras!")

---

## 9. Gerar Letras

### Parametros de Geracao

| Parametro | Descricao | Valor padrao |
|-----------|-----------|-------------|
| `genero` | Genero musical | `'Pagode'` |
| `titulo` | Titulo da musica (opcional) | `''` |
| `inicio` | Primeiras palavras da letra (opcional) | `''` |
| `temperature` | Criatividade (0.1=conservador, 1.5=ousado) | `0.9` |
| `top_k` | Limita a K tokens mais provaveis | `50` |
| `top_p` | Nucleus sampling (0.0-1.0) | `0.92` |
| `max_tokens` | Comprimento maximo da letra gerada | `400` |
| `num_letras` | Quantidade de letras para gerar | `1` |

In [ ]:
def gerar_letra(
    genero='Pagode',
    titulo='',
    inicio='',
    temperature=0.9,
    top_k=50,
    top_p=0.92,
    max_tokens=400,
    repetition_penalty=1.2,
    num_letras=1,
):
    """
    Gera letras de musica condicionadas por genero.

    Args:
        genero: 'MPB', 'Trap', 'Sertanejo', 'Pagode', 'Arrocha'
        titulo: Titulo da musica (opcional, modelo inventa se vazio)
        inicio: Primeiras palavras da letra (opcional)
        temperature: Criatividade (0.1 a 1.5)
        top_k: Top-K sampling
        top_p: Nucleus sampling
        max_tokens: Tokens maximos para gerar
        repetition_penalty: Penalidade por repeticao (1.0=sem penalidade)
        num_letras: Quantas letras gerar

    Returns:
        Lista de letras geradas
    """
    # Montar prompt condicionado
    prompt = f"<|genero|>{genero}"

    if titulo:
        prompt += f"<|titulo|>{titulo}"

    prompt += "<|letra|>\n"

    if inicio:
        prompt += inicio

    # Tokenizar prompt
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

    # Token de parada
    fim_token_id = tokenizer.convert_tokens_to_ids('<|fim|>')

    resultados = []

    for i in range(num_letras):
        with torch.no_grad():
            output = model.generate(
                input_ids,
                max_new_tokens=max_tokens,
                temperature=temperature,
                top_k=top_k,
                top_p=top_p,
                repetition_penalty=repetition_penalty,
                do_sample=True,
                num_return_sequences=1,
                eos_token_id=fim_token_id,
                pad_token_id=tokenizer.pad_token_id,
            )

        # Decodificar
        full_text = tokenizer.decode(output[0], skip_special_tokens=False)

        # Extrair apenas a letra (entre <|letra|> e <|fim|>)
        letra = full_text
        if '<|letra|>' in letra:
            letra = letra.split('<|letra|>')[-1]
        if '<|fim|>' in letra:
            letra = letra.split('<|fim|>')[0]

        # Limpar tokens especiais residuais
        for tok in ['<|genero|>', '<|titulo|>', '<|pad|>', '<|endoftext|>']:
            letra = letra.replace(tok, '')

        letra = letra.strip()
        resultados.append(letra)

    return resultados


def exibir_letra(letra, genero='', titulo='', numero=None):
    """Exibe uma letra formatada."""
    print("\n" + "=" * 60)
    header = ""
    if numero is not None:
        header += f"LETRA #{numero} "
    if genero:
        header += f"[{genero}]"
    if titulo:
        header += f" - {titulo}"
    print(f" {header} ".center(60, "="))
    print("=" * 60)
    print()
    print(letra)
    print()
    print("-" * 60)
    print(f"  ({len(letra.split())} palavras, {len(letra.splitlines())} linhas)")
    print()


print("Funcoes de geracao carregadas.")
print("\nGeneros disponiveis: MPB, Trap, Sertanejo, Pagode, Arrocha")

### 9.1 Gerar por Genero

In [ ]:
# ====================================================================
# GERAR LETRA - Escolha o genero e ajuste os parametros
# ====================================================================

GENERO = 'Pagode'       # MPB, Trap, Sertanejo, Pagode, Arrocha
TITULO = ''              # Deixe vazio para o modelo inventar
TEMPERATURA = 0.9        # 0.7=conservador, 0.9=equilibrado, 1.2=criativo
NUM_LETRAS = 3           # Quantas variantes gerar

letras = gerar_letra(
    genero=GENERO,
    titulo=TITULO,
    temperature=TEMPERATURA,
    num_letras=NUM_LETRAS,
)

for i, letra in enumerate(letras, 1):
    exibir_letra(letra, genero=GENERO, titulo=TITULO, numero=i)

### 9.2 Gerar com Titulo Especifico

In [ ]:
# Gerar com titulo definido
letras = gerar_letra(
    genero='Sertanejo',
    titulo='Saudade Que Doi',
    temperature=0.85,
    num_letras=2,
)

for i, letra in enumerate(letras, 1):
    exibir_letra(letra, genero='Sertanejo', titulo='Saudade Que Doi', numero=i)

### 9.3 Gerar com Inicio da Letra

In [ ]:
# Gerar continuando de um trecho inicial
letras = gerar_letra(
    genero='MPB',
    titulo='Janela Pro Mar',
    inicio='Da janela eu vejo o mar\nE o mar me leva pra longe\n',
    temperature=0.9,
    num_letras=2,
)

for i, letra in enumerate(letras, 1):
    exibir_letra(letra, genero='MPB', titulo='Janela Pro Mar', numero=i)

### 9.4 Comparar Generos

In [ ]:
# Gerar o mesmo tema em generos diferentes
TEMA_TITULO = 'Noite de Amor'

print("Gerando o mesmo tema em todos os generos...\n")

for genero in ['MPB', 'Pagode', 'Sertanejo', 'Arrocha', 'Trap']:
    letras = gerar_letra(
        genero=genero,
        titulo=TEMA_TITULO,
        temperature=0.9,
        num_letras=1,
    )
    exibir_letra(letras[0], genero=genero, titulo=TEMA_TITULO)

### 9.5 Modo Exploratoria - Variar Temperatura

In [ ]:
# Ver como a temperatura afeta a geracao
GENERO_TESTE = 'Trap'

print(f"Testando temperaturas diferentes para {GENERO_TESTE}:\n")

for temp in [0.5, 0.8, 1.0, 1.3]:
    letras = gerar_letra(
        genero=GENERO_TESTE,
        temperature=temp,
        max_tokens=200,
        num_letras=1,
    )
    print(f"--- Temperatura {temp} ---")
    # Mostrar apenas as primeiras 8 linhas
    linhas = letras[0].splitlines()[:8]
    for l in linhas:
        print(f"  {l}")
    print()

## 10. Gerador Interativo

Interface com formularios para gerar letras facilmente.

In [ ]:
# @title Gerador Interativo de Letras { run: "auto", form-width: "80%" }

genero = 'Pagode'  # @param ['MPB', 'Trap', 'Sertanejo', 'Pagode', 'Arrocha']
titulo = ''  # @param {type: "string"}
inicio_da_letra = ''  # @param {type: "string"}
temperatura = 0.9  # @param {type: "slider", min: 0.1, max: 1.5, step: 0.1}
top_k = 50  # @param {type: "slider", min: 10, max: 100, step: 10}
top_p = 0.92  # @param {type: "slider", min: 0.5, max: 1.0, step: 0.02}
penalidade_repeticao = 1.2  # @param {type: "slider", min: 1.0, max: 2.0, step: 0.1}
max_tokens = 400  # @param {type: "slider", min: 100, max: 600, step: 50}
quantidade = 1  # @param {type: "slider", min: 1, max: 5, step: 1}

letras = gerar_letra(
    genero=genero,
    titulo=titulo,
    inicio=inicio_da_letra,
    temperature=temperatura,
    top_k=top_k,
    top_p=top_p,
    max_tokens=max_tokens,
    repetition_penalty=penalidade_repeticao,
    num_letras=quantidade,
)

for i, letra in enumerate(letras, 1):
    exibir_letra(letra, genero=genero, titulo=titulo, numero=i)

## 11. Exportar Letras Geradas

In [ ]:
def salvar_letras(letras, genero, titulo='', filepath=None):
    """
    Salva letras geradas em arquivo texto.

    Args:
        letras: Lista de letras geradas
        genero: Genero das letras
        titulo: Titulo (opcional)
        filepath: Caminho para salvar (opcional, usa Drive por padrao)
    """
    if filepath is None:
        output_dir = os.path.join(DRIVE_BASE, 'letras_geradas')
        os.makedirs(output_dir, exist_ok=True)

        # Nome do arquivo baseado no genero e timestamp
        from datetime import datetime
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        safe_titulo = titulo.replace(' ', '_')[:30] if titulo else 'sem_titulo'
        filename = f"{genero}_{safe_titulo}_{timestamp}.txt"
        filepath = os.path.join(output_dir, filename)

    with open(filepath, 'w', encoding='utf-8') as f:
        for i, letra in enumerate(letras, 1):
            f.write(f"### Letra #{i} [{genero}]")
            if titulo:
                f.write(f" - {titulo}")
            f.write("\n\n")
            f.write(letra)
            f.write("\n\n")
            f.write("=" * 60)
            f.write("\n\n")

    print(f"Letras salvas em: {filepath}")
    return filepath


# Exemplo: salvar as ultimas letras geradas
# salvar_letras(letras, genero=genero, titulo=titulo)

## 12. Dicas de Uso

### Temperatura
- **0.5-0.7**: Letras conservadoras, mais proximas do corpus. Bom para manter fidelidade ao genero.
- **0.8-1.0**: Equilibrio entre fidelidade e criatividade. Recomendado para a maioria dos casos.
- **1.1-1.5**: Letras mais experimentais e inesperadas. Pode gerar neologismos.

### Generos
- **MPB**: Letras poeticas, vocabulario sofisticado (~179 palavras/musica)
- **Trap**: Letras longas, mix de idiomas (~418 palavras/musica)
- **Sertanejo**: Tematica de amor/sofrencia, refroes marcantes (~240 palavras/musica)
- **Pagode**: Romance, emocao, vocabulario acessivel (~232 palavras/musica)
- **Arrocha**: Amor/sofrimento, estrutura direta (~182 palavras/musica)

### Penalidade de Repeticao
- **1.0**: Sem penalidade (permite repeticoes naturais como refroes)
- **1.2**: Penalidade leve (recomendado - reduz repeticao excessiva)
- **1.5+**: Penalidade forte (forca vocabulario diverso, pode perder naturalidade)

### Inicio da Letra
- Forneca as primeiras linhas para direcionar o tema
- Exemplo: `'Hoje eu acordei pensando em voce\n'`
- O modelo vai continuar no estilo e tema sugerido

### Workflow Recomendado
1. Gere 3-5 variantes com `num_letras=3`
2. Escolha os melhores trechos de cada
3. Combine e edite manualmente
4. Use `inicio` para regenerar secoes especificas

---

Corpus: 5.159 musicas | Generos: MPB, Trap, Sertanejo, Pagode, Arrocha | Modelo base: GPT-2 Portuguese